In [1]:
import os
import numpy as np
from cffi import FFI
ffi = FFI()
ffi.cdef("""
    void* create_rosenblatt_classifier(
        size_t n_features,
        float learning_rate,
        float bias_cat,
        float bias_lion,
        float bias_cheetah,
        uint64_t seed
    );

    void train_classifier(
        void* classifier,
        const float* x,
        size_t rows,
        size_t cols,
        const size_t* y,
        size_t y_len,
        size_t epochs
    );

    size_t* predict_classifier(
        void* classifier,
        const float* x,
        size_t rows,
        size_t cols,
        const size_t* y,
        size_t y_len
    );
    typedef struct {
        float* data;
        size_t rows;
        size_t cols;
    } MatrixFFI;
    MatrixFFI* transforming(void* classifier, const float* x, size_t rows, size_t cols);
    void release_matrix_ffi(MatrixFFI* matrix);
    
""", override=True)

dll_path = os.path.abspath("../felidae_classifier/target/debug/felidae_classifier.dll")
lib = ffi.dlopen(dll_path)

classifier = lib.create_rosenblatt_classifier(
    2,
    1.0,
    1.1,
    1.0,
    1.0,
    200
)

import numpy as np

X_linear = np.ascontiguousarray([
    
    [0.1, 0.1], [0.2, 0.15], [0.15, 0.2], [0.1, 0.25], [0.25, 0.1],
    
    [0.5, 0.9], [0.55, 0.85], [0.45, 0.88], [0.5, 0.95], [0.6, 0.9],
    
    [0.9, 0.1], [0.85, 0.15], [0.88, 0.2], [0.95, 0.1], [0.9, 0.25],
], dtype=np.float32)

Y_linear = np.ascontiguousarray([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    2, 2, 2, 2, 2,
], dtype=np.uintp)

x_ptr = ffi.from_buffer("float[]", X_linear)
y_ptr = ffi.from_buffer("size_t[]", Y_linear)

lib.train_classifier(
    classifier,
    x_ptr,
    X_linear.shape[0],
    X_linear.shape[1],
    y_ptr,
    Y_linear.shape[0],
    100
)
test = lib.predict_classifier(classifier,
    x_ptr,
    X_linear.shape[0],
    X_linear.shape[1],
    y_ptr,
    Y_linear.shape[0]
)
print("test", test)
pred = np.frombuffer(
    ffi.buffer(test, X_linear.shape[0] * ffi.sizeof("size_t")),
    dtype=np.uintp
)
print(pred)




X_nonlinear = np.ascontiguousarray([
    [0.1, 0.9], [0.2, 0.8], [0.15, 0.85],
    [0.9, 0.1], [0.8, 0.2], [0.85, 0.15],
    
    [0.9, 0.9], [0.8, 0.8], [0.85, 0.85],
    [0.1, 0.1], [0.2, 0.2], [0.15, 0.15],
    
    [0.5, 0.5], [0.55, 0.45], [0.45, 0.55],
    [0.5, 0.6], [0.6, 0.5],
], dtype=np.float32)

Y_nonlinear = np.ascontiguousarray([
    0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1,
    2, 2, 2, 2, 2,
], dtype=np.uintp)

x_nonlinear_ptr = ffi.from_buffer("float[]", X_nonlinear)
y_nonlinear_ptr = ffi.from_buffer("size_t[]", Y_nonlinear)

result = lib.transforming(classifier, x_nonlinear_ptr, X_nonlinear.shape[0], X_nonlinear.shape[1])
data = np.frombuffer(
    ffi.buffer(result.data, result.rows * result.cols * ffi.sizeof("float")),
    dtype=np.float32
).reshape(result.rows, result.cols)

print("Transformed shape:", data.shape)
print("Transformed data:\n", data)


test <cdata 'size_t *' 0x0000021B2BB254C0>
[0 0 0 0 0 1 1 1 1 1 2 2 2 2 2]
Transformed shape: (17, 5)
Transformed data:
 [[0.1        0.9        0.01       0.09       0.80999994]
 [0.2        0.8        0.04       0.16000001 0.64000005]
 [0.15       0.85       0.0225     0.12750001 0.7225    ]
 [0.9        0.1        0.80999994 0.09       0.01      ]
 [0.8        0.2        0.64000005 0.16000001 0.04      ]
 [0.85       0.15       0.7225     0.12750001 0.0225    ]
 [0.9        0.9        0.80999994 0.80999994 0.80999994]
 [0.8        0.8        0.64000005 0.64000005 0.64000005]
 [0.85       0.85       0.7225     0.7225     0.7225    ]
 [0.1        0.1        0.01       0.01       0.01      ]
 [0.2        0.2        0.04       0.04       0.04      ]
 [0.15       0.15       0.0225     0.0225     0.0225    ]
 [0.5        0.5        0.25       0.25       0.25      ]
 [0.55       0.45       0.3025     0.2475     0.20249999]
 [0.45       0.55       0.20249999 0.2475     0.3025    ]
 [0.5    